# MCP 第 05 课：让本地 LLM 通过 MCP 使用工具

到这里，MCP 的 server 和 client 我们都已经有了。

最后一步就是回答一个非常实际的问题：

> 怎么让本地部署的 LLM，真正通过 MCP 去使用工具？


## 这节课的策略

一个常见做法是：

1. 用 MCP client 连接 server，拿到工具描述
2. 把这些工具描述转换成 OpenAI tool schema
3. 发给本地 OpenAI 兼容模型
4. 如果模型返回 tool call，就再通过 MCP client 真正执行
5. 把结果送回模型，拿最终答案

这本质上就是“LLM + MCP bridge”。


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/siji/Desktop/AI/mcp/.env")

BASE_URL = os.getenv("OPENAI_BASE_URL", "http://10.0.0.63:1234/v1")
API_KEY = os.getenv("OPENAI_API_KEY", "lm-studio")
MODEL = os.getenv("OPENAI_MODEL", "qwen/qwen3.6-35b-a3b")

BASE_URL, MODEL


In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
[m.id for m in client.models.list().data][:10]


## 把 MCP tools 转成 OpenAI tools

如果一个本地模型支持 OpenAI 风格的 tool calling，这一步就能把 MCP server 暴露的能力“翻译”给它。


In [ ]:
import asyncio
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PATH = Path("/Users/siji/Desktop/AI/mcp/src/mcp_course/server.py")


def to_openai_tools(mcp_tools):
    out = []
    for tool in mcp_tools:
        out.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": tool.inputSchema,
                },
            }
        )
    return out


async def fetch_tools():
    params = StdioServerParameters(command="python", args=[str(SERVER_PATH)])
    async with stdio_client(params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            return tools.tools


mcp_tools = asyncio.run(fetch_tools())
openai_tools = to_openai_tools(mcp_tools)
openai_tools[0]


## 看完整桥接实现

配套文件已经放在：

`src/mcp_course/llm_bridge.py`

它做的事情就是：

- 连上 MCP server
- 取出 tools
- 发给本地 LLM
- 执行模型请求的 tool call
- 再把结果发回模型


In [ ]:
from pathlib import Path

bridge_path = Path("/Users/siji/Desktop/AI/mcp/src/mcp_course/llm_bridge.py")
print(bridge_path)
print(bridge_path.read_text()[:1800])


## 本课工程直觉

1. **MCP 解决的是“能力标准化暴露”，LLM 解决的是“何时该用这些能力”。**
2. **bridge 的本质是翻译层：MCP schema -> LLM 能理解的 tool schema。**
3. **如果你的本地模型 tool calling 不稳定，也可以退回文本协议，但 MCP server 仍然有价值。**
